In [1]:
import os
import glob
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from tqdm import tqdm

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

2026-03-25 11:04:23.487293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774436663.689374      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774436663.744049      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774436664.194353      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774436664.194397      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774436664.194400      23 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla P100-PCIE-16GB


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
TEST_CSV    = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
MASHUP_DIR  = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
ESC50_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio'
OUTPUT_DIR  = '/kaggle/working'

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}

# ── Hyperparameters ───────────────────────────────────────────────────────────
SR          = 16000
BATCH_SIZE  = 8          # increase if GPU memory allows
NUM_WORKERS = 2

# Cross-genre mixing probability and blend strength
MASHUP_PROB     = 0.65   # how often to inject a foreign genre stem
MASHUP_STRENGTH = (0.15, 0.45)  # random blend range

# ESC-50 noise injection (keep mild)
ESC_PROB     = 0.40
ESC_STRENGTH = (0.01, 0.05)

# Label smoothing
LABEL_SMOOTHING = 0.10

# Mixup
MIXUP_ALPHA = 0.30
MIXUP_PROB  = 0.50

# TTA crops at inference
TTA_CROPS = 7
CROP_SECS = 8   # seconds per crop (AST default input is ~10 s)

print('Config loaded.')

Config loaded.


In [3]:
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(GENRES),
    ignore_mismatched_sizes=True
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded — {total_params/1e6:.1f}M parameters')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded — 86.2M parameters


In [4]:
song_paths, labels = [], []

for label, genre in enumerate(GENRES):
    genre_path = os.path.join(BASE_PATH, genre)
    for folder in os.listdir(genre_path):
        full_path = os.path.join(genre_path, folder)
        if os.path.isdir(full_path):
            song_paths.append(full_path)
            labels.append(label)

print(f'Total samples: {len(song_paths)}')

# ESC-50 noise files
esc50_files = glob.glob(os.path.join(ESC50_DIR, '*.wav'))
print(f'ESC-50 files:  {len(esc50_files)}')

# Collect ALL stems across every genre (used for cross-genre mixing)
all_stems = []
for path in song_paths:
    for f in os.listdir(path):
        if f.endswith('.wav'):
            all_stems.append(os.path.join(path, f))
print(f'Total stems available for mashup mixing: {len(all_stems)}')

# Train / val split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    song_paths, labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)
print(f'Train: {len(train_paths)}  |  Val: {len(val_paths)}')

Total samples: 1000
ESC-50 files:  2000
Total stems available for mashup mixing: 4000
Train: 800  |  Val: 200


In [5]:
def load_and_mix_stems(folder_path, random_select=True):
    """Load all stems from a folder, optionally randomly dropping some."""
    wav_files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
    if random_select and len(wav_files) > 2:
        k = random.randint(2, len(wav_files))
        wav_files = random.sample(wav_files, k)

    signals, min_len = [], None
    for f in wav_files:
        audio, _ = librosa.load(os.path.join(folder_path, f), sr=SR)
        if min_len is None:
            min_len = len(audio)
        else:
            min_len = min(min_len, len(audio))
        signals.append(audio)

    signals = [s[:min_len] for s in signals]
    return np.mean(signals, axis=0)


class ASTDataset(Dataset):
    """
    Key improvement: inject stems from OTHER genres to simulate mashup audio,
    matching the test set distribution.
    """
    def __init__(self, song_paths, labels, all_stems, esc50_files, train=True):
        self.song_paths  = song_paths
        self.labels      = labels
        self.all_stems   = all_stems
        self.esc50_files = esc50_files
        self.train       = train

    def __len__(self):
        return len(self.song_paths)

    def __getitem__(self, idx):
        path  = self.song_paths[idx]
        label = self.labels[idx]

        # ── 1. Mix own stems ─────────────────────────────────────────────────
        mixed = load_and_mix_stems(path, random_select=self.train)

        # ── 2. Cross-genre injection (TRAIN ONLY) ────────────────────────────
        #   This is the most important fix: simulate mashup audio by blending
        #   a stem from a different genre into the training sample.
        if self.train and random.random() < MASHUP_PROB:
            foreign_path = random.choice(self.all_stems)
            # Avoid using a stem from the same folder
            while path in foreign_path:
                foreign_path = random.choice(self.all_stems)

            foreign, _ = librosa.load(foreign_path, sr=SR)
            # Align lengths
            if len(foreign) < len(mixed):
                foreign = np.pad(foreign, (0, len(mixed) - len(foreign)))
            else:
                foreign = foreign[:len(mixed)]

            strength = random.uniform(*MASHUP_STRENGTH)
            mixed = mixed + strength * foreign

        # ── 3. ESC-50 background noise (mild, TRAIN ONLY) ────────────────────
        if self.train and self.esc50_files and random.random() < ESC_PROB:
            noise, _ = librosa.load(random.choice(self.esc50_files), sr=SR)
            if len(noise) < len(mixed):
                noise = np.pad(noise, (0, len(mixed) - len(noise)))
            else:
                noise = noise[:len(mixed)]
            strength = random.uniform(*ESC_STRENGTH)
            mixed = mixed + strength * noise

        # ── 4. Time-shift augmentation (TRAIN ONLY) ──────────────────────────
        if self.train and random.random() < 0.4:
            shift = random.randint(-SR * 2, SR * 2)
            mixed = np.roll(mixed, shift)

        # ── 5. Feature extraction ─────────────────────────────────────────────
        inputs = feature_extractor(
            mixed, sampling_rate=SR, return_tensors='pt'
        )
        return inputs['input_values'].squeeze(0), label


train_dataset = ASTDataset(train_paths, train_labels, all_stems, esc50_files, train=True)
val_dataset   = ASTDataset(val_paths,   val_labels,   all_stems, esc50_files, train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print('Datasets ready.')
sample_x, sample_y = train_dataset[0]
print(f'Sample input shape: {sample_x.shape}  |  label: {GENRES[sample_y]}')

Datasets ready.
Sample input shape: torch.Size([1024, 128])  |  label: jazz


In [6]:
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    """Apply mixup augmentation to a batch."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print('Loss function: CrossEntropyLoss with label_smoothing =', LABEL_SMOOTHING)

Loss function: CrossEntropyLoss with label_smoothing = 0.1


In [7]:
def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    """Linear warmup then cosine annealing."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(warmup_epochs)
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, use_mixup=True):
    model.train()
    total_loss = 0

    for batch_inputs, batch_labels in loader:
        batch_inputs  = batch_inputs.to(device)
        batch_labels  = batch_labels.to(device)

        optimizer.zero_grad()

        if use_mixup and random.random() < MIXUP_PROB:
            mixed_x, y_a, y_b, lam = mixup_data(batch_inputs, batch_labels)
            outputs = model(mixed_x)
            loss = mixup_criterion(criterion, outputs.logits, y_a, y_b, lam)
        else:
            outputs = model(batch_inputs)
            loss = criterion(outputs.logits, batch_labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for batch_inputs, batch_labels in loader:
        batch_inputs = batch_inputs.to(device)
        outputs = model(batch_inputs)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(batch_labels.numpy())

    return f1_score(all_true, all_preds, average='macro'), all_preds, all_true


def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              epochs, phase_name, use_mixup=True):
    print(f'\n{'='*60}')
    print(f'  {phase_name}')
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable parameters: {trainable/1e6:.2f}M')
    print(f'{'='*60}')

    best_f1, best_state = 0, None

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, use_mixup)
        val_f1, _, _ = evaluate(model, val_loader)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        print(f'  Epoch {epoch+1:2d}/{epochs} | '
              f'Loss {train_loss:.4f} | '
              f'Val F1 {val_f1:.4f} | '
              f'LR {lr:.2e}'
              + (' ★' if val_f1 > best_f1 else ''))

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'\n  Best val F1 in {phase_name}: {best_f1:.4f}')
    model.load_state_dict(best_state)
    return model, best_f1


print('Training utilities ready.')

Training utilities ready.


In [8]:
# ── Phase 1: Classifier head only ────────────────────────────────────────────
for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

optimizer1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4, weight_decay=1e-4
)
scheduler1 = get_warmup_cosine_scheduler(optimizer1, warmup_epochs=1, total_epochs=5)

model, p1_f1 = run_phase(
    model, train_loader, val_loader, optimizer1, scheduler1,
    epochs=5, phase_name='Phase 1 — Classifier head', use_mixup=False
)


  Phase 1 — Classifier head
  Trainable parameters: 0.01M
  Epoch  1/5 | Loss 1.9070 | Val F1 0.7343 | LR 3.00e-04 ★
  Epoch  2/5 | Loss 1.4110 | Val F1 0.7714 | LR 2.56e-04 ★
  Epoch  3/5 | Loss 1.3449 | Val F1 0.8078 | LR 1.50e-04 ★
  Epoch  4/5 | Loss 1.2404 | Val F1 0.8105 | LR 4.39e-05 ★
  Epoch  5/5 | Loss 1.2612 | Val F1 0.8249 | LR 0.00e+00 ★

  Best val F1 in Phase 1 — Classifier head: 0.8249


In [9]:
# ── Phase 2: Head + last 6 transformer layers ─────────────────────────────────
UNFREEZE_LAYERS_P2 = [str(i) for i in range(6, 12)]  # layers 6-11

for name, param in model.audio_spectrogram_transformer.named_parameters():
    if any(f'encoder.layer.{i}.' in name for i in UNFREEZE_LAYERS_P2):
        param.requires_grad = True

optimizer2 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-6, weight_decay=1e-4
)
scheduler2 = get_warmup_cosine_scheduler(optimizer2, warmup_epochs=2, total_epochs=10)

model, p2_f1 = run_phase(
    model, train_loader, val_loader, optimizer2, scheduler2,
    epochs=10, phase_name='Phase 2 — Last 6 transformer layers'
)


  Phase 2 — Last 6 transformer layers
  Trainable parameters: 42.54M
  Epoch  1/10 | Loss 1.3402 | Val F1 0.8255 | LR 5.00e-06 ★
  Epoch  2/10 | Loss 1.2706 | Val F1 0.8599 | LR 5.00e-06 ★
  Epoch  3/10 | Loss 1.3546 | Val F1 0.8407 | LR 4.81e-06
  Epoch  4/10 | Loss 1.2778 | Val F1 0.8693 | LR 4.27e-06 ★
  Epoch  5/10 | Loss 1.1708 | Val F1 0.8698 | LR 3.46e-06 ★
  Epoch  6/10 | Loss 1.2490 | Val F1 0.8685 | LR 2.50e-06
  Epoch  7/10 | Loss 1.1925 | Val F1 0.8683 | LR 1.54e-06
  Epoch  8/10 | Loss 1.2105 | Val F1 0.8587 | LR 7.32e-07
  Epoch  9/10 | Loss 1.1708 | Val F1 0.8583 | LR 1.90e-07
  Epoch 10/10 | Loss 1.1548 | Val F1 0.8682 | LR 0.00e+00

  Best val F1 in Phase 2 — Last 6 transformer layers: 0.8698


In [10]:
# ── Phase 3: Full fine-tune ───────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = True

optimizer3 = torch.optim.AdamW(
    model.parameters(), lr=1e-6, weight_decay=1e-4
)
scheduler3 = get_warmup_cosine_scheduler(optimizer3, warmup_epochs=2, total_epochs=10)

model, p3_f1 = run_phase(
    model, train_loader, val_loader, optimizer3, scheduler3,
    epochs=10, phase_name='Phase 3 — Full fine-tune'
)

print(f'\nPhase F1 summary  →  P1: {p1_f1:.4f}  |  P2: {p2_f1:.4f}  |  P3: {p3_f1:.4f}')


  Phase 3 — Full fine-tune
  Trainable parameters: 86.20M
  Epoch  1/10 | Loss 1.1618 | Val F1 0.8538 | LR 1.00e-06 ★
  Epoch  2/10 | Loss 1.2486 | Val F1 0.8737 | LR 1.00e-06 ★
  Epoch  3/10 | Loss 1.1941 | Val F1 0.8828 | LR 9.62e-07 ★
  Epoch  4/10 | Loss 1.1834 | Val F1 0.8781 | LR 8.54e-07
  Epoch  5/10 | Loss 1.1977 | Val F1 0.8744 | LR 6.91e-07
  Epoch  6/10 | Loss 1.1807 | Val F1 0.8737 | LR 5.00e-07
  Epoch  7/10 | Loss 1.1893 | Val F1 0.8789 | LR 3.09e-07
  Epoch  8/10 | Loss 1.1317 | Val F1 0.8733 | LR 1.46e-07
  Epoch  9/10 | Loss 1.1500 | Val F1 0.8733 | LR 3.81e-08
  Epoch 10/10 | Loss 1.1369 | Val F1 0.8733 | LR 0.00e+00

  Best val F1 in Phase 3 — Full fine-tune: 0.8828

Phase F1 summary  →  P1: 0.8249  |  P2: 0.8698  |  P3: 0.8828


In [11]:
# ── Save best checkpoint ─────────────────────────────────────────────────────
ckpt_path = os.path.join(OUTPUT_DIR, 'ast_best.pt')
torch.save(model.state_dict(), ckpt_path)
print(f'Model saved → {ckpt_path}')

# Validation classification report
_, val_preds, val_true = evaluate(model, val_loader)
print('\nFinal validation classification report:')
print(classification_report(val_true, val_preds, target_names=GENRES))

Model saved → /kaggle/working/ast_best.pt

Final validation classification report:
              precision    recall  f1-score   support

       blues       0.86      0.90      0.88        20
   classical       1.00      1.00      1.00        20
     country       0.76      0.80      0.78        20
       disco       0.86      0.95      0.90        20
      hiphop       1.00      0.85      0.92        20
        jazz       0.95      0.95      0.95        20
       metal       0.86      0.95      0.90        20
         pop       0.81      0.85      0.83        20
      reggae       0.95      1.00      0.98        20
        rock       0.80      0.60      0.69        20

    accuracy                           0.89       200
   macro avg       0.89      0.89      0.88       200
weighted avg       0.89      0.89      0.88       200



In [12]:
CROP_SAMPLES = CROP_SECS * SR  # samples per TTA crop


@torch.no_grad()
def predict_with_tta(model, audio, n_crops=TTA_CROPS):
    """
    Multi-crop TTA: take n random crops, average the logits.
    Falls back to the full clip when audio is shorter than one crop.
    """
    model.eval()
    logits_list = []
    length = len(audio)

    for i in range(n_crops):
        if length <= CROP_SAMPLES:
            crop = audio  # clip is already short — just use it
        else:
            if i == 0:
                # First crop: centre
                start = (length - CROP_SAMPLES) // 2
            else:
                start = random.randint(0, length - CROP_SAMPLES)
            crop = audio[start: start + CROP_SAMPLES]

        inputs = feature_extractor(crop, sampling_rate=SR, return_tensors='pt')
        x = inputs['input_values'].to(device)  # (1, time, freq)
        logits = model(x).logits.cpu()          # (1, num_classes)
        logits_list.append(logits)

    avg_logits = torch.mean(torch.stack(logits_list), dim=0)  # (1, num_classes)
    return torch.argmax(avg_logits, dim=1).item()


print(f'TTA ready — {TTA_CROPS} crops × {CROP_SECS}s each')

TTA ready — 7 crops × 8s each


In [13]:
test_df = pd.read_csv(TEST_CSV)
print(f'Test samples: {len(test_df)}')
print(test_df.head())

Test samples: 3020
   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav


In [14]:
all_ids, all_preds = [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Inferring'):
    file_path = os.path.join(MASHUP_DIR, row['filename'])

    audio, _ = librosa.load(file_path, sr=SR)

    pred_idx = predict_with_tta(model, audio, n_crops=TTA_CROPS)

    all_ids.append(int(row['id']))
    all_preds.append(GENRES[pred_idx])


submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(submission_path, index=False)

print(f'\nSubmission saved → {submission_path}')
print(f'Total predictions: {len(submission)}')
print('\nPrediction distribution:')
print(submission['genre'].value_counts())

Inferring: 100%|██████████| 3020/3020 [20:46<00:00,  2.42it/s]


Submission saved → /kaggle/working/submission.csv
Total predictions: 3020

Prediction distribution:
genre
hiphop       413
pop          400
reggae       341
blues        313
metal        305
jazz         296
classical    272
country      265
disco        228
rock         187
Name: count, dtype: int64


In [15]:
# # ── Ensemble template (run after training a second model) ─────────────────────
# # Uncomment and fill in model2 path when ready.

# model2 = ASTForAudioClassification.from_pretrained(
#     MODEL_NAME, num_labels=len(GENRES), ignore_mismatched_sizes=True
# )
# model2.load_state_dict(torch.load('/kaggle/working/ast_best_run2.pt', map_location=device))
# model2.to(device)

# @torch.no_grad()
# def predict_ensemble(models, audio, weights=None, n_crops=TTA_CROPS):
#     if weights is None:
#         weights = [1.0 / len(models)] * len(models)
#     combined = None
#     for m, w in zip(models, weights):
#         m.eval()
#         logits_list = []
#         for _ in range(n_crops):
#             start = random.randint(0, max(0, len(audio) - CROP_SAMPLES))
#             crop = audio[start:start + CROP_SAMPLES]
#             inputs = feature_extractor(crop, sampling_rate=SR, return_tensors='pt')
#             x = inputs['input_values'].to(device)
#             logits_list.append(m(x).logits.cpu())
#         avg = torch.mean(torch.stack(logits_list), dim=0)
#         combined = w * avg if combined is None else combined + w * avg
#     return torch.argmax(combined, dim=1).item()

# # Re-run inference with ensemble
# all_ids, all_preds = [], []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Ensemble inference'):
#     audio, _ = librosa.load(os.path.join(MASHUP_DIR, row['filename']), sr=SR)
#     pred_idx = predict_ensemble([model, model2], audio, weights=[0.55, 0.45])
#     all_ids.append(int(row['id']))
#     all_preds.append(GENRES[pred_idx])
# submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
# submission.to_csv(os.path.join(OUTPUT_DIR, 'submission_ensemble.csv'), index=False)
# print('Ensemble submission saved.')

# print('Ensemble template ready — uncomment when you have a second model checkpoint.')